In [3]:
import os
import numpy as np
import cv2
import h5py
import pickle
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# --- Load and Preprocess Images ---
def load_images_from_directory(directory, target_size=(64, 64)):
    """Load images from a directory, resize them, and return as arrays."""
    images = []
    labels = []
    class_names = sorted(os.listdir(directory))  
    class_mapping = {name: idx for idx, name in enumerate(class_names)}
    
    for class_name in class_names:
        class_dir = os.path.join(directory, class_name)
        if not os.path.isdir(class_dir):
            continue
        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = cv2.resize(img, target_size)
            images.append(img)
            labels.append(class_mapping[class_name])
    
    return np.array(images), np.array(labels), class_mapping

# Load train, validation, and test sets
train_images, train_labels, class_mapping = load_images_from_directory(r'E:\New folder (2)\New folder\LiteCNN4Rice\train')
val_images, val_labels, _ = load_images_from_directory(r'E:\New folder (2)\New folder\LiteCNN4Rice\val')
test_images, test_labels, _ = load_images_from_directory(r'E:\New folder (2)\New folder\LiteCNN4Rice\test')

# Merge train and validation sets
combined_images = np.concatenate((train_images, val_images), axis=0)
combined_labels = np.concatenate((train_labels, val_labels), axis=0)

# Normalize pixel values (0-1)
combined_images = combined_images / 255.0
test_images = test_images / 255.0

# Flatten image data for ML models
x_train_combined = combined_images.reshape(combined_images.shape[0], -1)
x_test = test_images.reshape(test_images.shape[0], -1)

# Define models
models = {
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Support Vector Machine": SVC(random_state=42),

}



# --- Train and Evaluate ML Models ---
for model_name, model in models.items():
    print(f"\n--- {model_name} ---")
    model.fit(x_train_combined, combined_labels)
    
    # Predict on test set
    test_predictions = model.predict(x_test)
    
    # Evaluate
    test_accuracy = accuracy_score(test_labels, test_predictions)
    print(f"Test Accuracy: {test_accuracy}")
    print("Test Performance:")
    print(classification_report(test_labels, test_predictions, target_names=list(class_mapping.keys())))
    
    # Save the model using h5py
    model_path = os.path.join(f"{model_name.replace(' ', '_')}.h5")
    with h5py.File(model_path, 'w') as h5file:
        # Serialize the model using pickle
        serialized_model = pickle.dumps(model)
        h5file.create_dataset("model", data=np.void(serialized_model))
    
    print(f"Model saved to: {model_path}")

print("\nAll models have been trained and saved.")



--- K-Nearest Neighbors ---
Test Accuracy: 0.7176470588235294
Test Performance:
                 precision    recall  f1-score   support

Bacterialblight       0.92      0.58      0.72       159
          Blast       0.70      0.60      0.65       144
      Brownspot       0.61      0.96      0.74       160
         Tungro       0.80      0.71      0.76       132

       accuracy                           0.72       595
      macro avg       0.76      0.71      0.71       595
   weighted avg       0.76      0.72      0.71       595

Model saved to: K-Nearest_Neighbors.h5

--- Support Vector Machine ---
Test Accuracy: 0.9680672268907563
Test Performance:
                 precision    recall  f1-score   support

Bacterialblight       0.96      0.98      0.97       159
          Blast       0.96      0.94      0.95       144
      Brownspot       0.96      0.99      0.98       160
         Tungro       0.99      0.95      0.97       132

       accuracy                           0.97    